# Zadanie 2 — NumPy: macierze, osie, broadcasting i losowość

**Czas:** 30–35 minut  
**Poziom:** podstawowy/średni  
**Cel:** analizować macierz zużycia energii, w której wiersze oznaczają budynki, a kolumny kolejne okresy.

Wartości są syntetyczne. Użyj operacji na całych tablicach zamiast ręcznego liczenia wartości w pętlach.


> **WERSJA DLA PROWADZĄCEGO — kompletne rozwiązanie.**
>
> Uruchom komórki od góry. Kod, wyniki kontrolne i interpretacje są częścią odpowiedzi. Dane są syntetyczne.


In [ ]:
import numpy as np

ids = np.array(["A01", "A02", "A03", "A04", "A05", "A06"])
areas = np.array([40., 65., 82., 55., 120., 73.])
periods = np.array(["I", "II", "III", "IV"])

usage = np.array([
    [180., 160., 170., 210.],
    [293., 260., 275., 340.],
    [369., 330., 350., 410.],
    [250., 230., 245., 290.],
    [620., 580., 610., 720.],
    [340., 310., 325., 390.],
])

print("Kształt macierzy:", usage.shape)


## A. Operacje wzdłuż osi

1. Sprawdź `usage.ndim` i `usage.shape`. Napisz, co oznaczają obie liczby kształtu.
2. Utwórz `total_by_building`: suma czterech okresów dla każdego budynku. Wybierz odpowiednią oś funkcji `np.sum`.
3. Utwórz `mean_by_period`: średnia z sześciu budynków w każdym okresie.
4. Znajdź identyfikator budynku o największej sumie oraz nazwę okresu o największej średniej.


In [ ]:
print("Liczba wymiarów:", usage.ndim)
print("Kształt:", usage.shape)  # 6 budynków × 4 okresy

total_by_building = np.sum(usage, axis=1)
mean_by_period = np.mean(usage, axis=0)

largest_total_id = ids[np.argmax(total_by_building)]
highest_mean_period = periods[np.argmax(mean_by_period)]

print("Suma dla budynków:", total_by_building)
print("Średnia dla okresów:", np.round(mean_by_period, 2))
print("Największa suma:", largest_total_id)
print("Największa średnia:", highest_mean_period)


## B. Broadcasting

5. Oblicz `intensity`, czyli zużycie w każdym okresie podzielone przez metraż właściwego budynku. Wynik ma mieć kształt `(6, 4)`. Zwróć uwagę na kształt `areas[:, None]`.
6. Zastosuj wektor korekt `[1.10, 0.90, 1.00, 1.20]` do odpowiednich kolumn `usage`. Zapisz nową macierz jako `adjusted`.
7. Sprawdź kształt obu wyników i wyświetl pierwszy wiersz każdej macierzy.


In [ ]:
period_factors = np.array([1.10, 0.90, 1.00, 1.20])

print("Kształt areas[:, None]:", areas[:, None].shape)  # (6, 1)
intensity = usage / areas[:, None]
adjusted = usage * period_factors

print("Kształt intensity:", intensity.shape)
print("Pierwszy budynek, kWh/m²:", np.round(intensity[0], 2))
print("Kształt adjusted:", adjusted.shape)
print("Pierwszy budynek po korekcie:", adjusted[0])


## C. Powtarzalna symulacja

8. Utwórz `rng = np.random.default_rng(42)` i macierz zakłóceń `noise` z rozkładu normalnego o średniej 0, odchyleniu standardowym 5 i kształcie `usage.shape`.
9. Oblicz `simulated = usage + noise`.
10. Uruchom ponownie komórkę i sprawdź, czy wynik jest taki sam. Wyjaśnij w dwóch zdaniach, co daje ustawienie ziarna, a czego samo ziarno nie gwarantuje.


In [ ]:
rng = np.random.default_rng(42)
noise = rng.normal(loc=0, scale=5, size=usage.shape)
simulated = usage + noise

print("Pierwszy wiersz zakłóceń:", np.round(noise[0], 2))
print("Pierwszy wiersz symulacji:", np.round(simulated[0], 2))

# Niezależnie od wcześniejszego stanu generatora sprawdzamy odtworzenie:
rng_check = np.random.default_rng(42)
noise_check = rng_check.normal(loc=0, scale=5, size=usage.shape)
print("Powtarzalne losowanie:", np.array_equal(noise, noise_check))


### Interpretacja prowadzącego

1. Ustawienie ziarna sprawia, że **nowy generator utworzony z tym samym ziarnem** wygeneruje tę samą sekwencję przy tych samych wywołaniach. Samo ponowne wywołanie `rng.normal(...)` na istniejącym generatorze da następny fragment sekwencji.
2. Do odtworzenia całego eksperymentu potrzebujemy również danych, kodu, kolejności operacji i informacji o środowisku, w tym wersji bibliotek.


## Samokontrola

Uruchom po wykonaniu zadań A–C.


In [ ]:
assert usage.shape == (6, 4)
assert total_by_building.shape == (6,)
assert mean_by_period.shape == (4,)
assert ids[np.argmax(total_by_building)] == "A05"
assert periods[np.argmax(mean_by_period)] == "IV"
assert intensity.shape == adjusted.shape == simulated.shape == (6, 4)
assert np.isclose(intensity[0, 0], 4.5)
assert np.isclose(adjusted[0, 0], 198.0)
assert np.allclose(simulated - usage, noise)
print("Kontrola liczbowa zakończona pomyślnie.")


## Wyzwanie dodatkowe

Znormalizuj każdy wiersz przez średnią z czterech okresów dla danego budynku. Które okresy przekraczają średnią danego budynku o co najmniej 10%?


## Rozwiązanie wyzwania dodatkowego

Średnią liczymy osobno dla każdego budynku (`axis=1`) i zachowujemy wymiar `(6, 1)`, aby dzielenie przez macierz `(6, 4)` działało przez broadcasting.


In [ ]:
row_mean = usage.mean(axis=1, keepdims=True)
relative_to_mean = usage / row_mean
over_10_percent = relative_to_mean >= 1.10

for i, j in np.argwhere(over_10_percent):
    print(ids[i], "okres", periods[j],
          f"{relative_to_mean[i, j]:.2f} × średnia tego budynku")

assert relative_to_mean.shape == usage.shape


**Wniosek:** porównujemy każdy okres z własnym poziomem odniesienia budynku. Dzięki temu duży budynek nie dominuje jedynie dlatego, że zużywa więcej kWh w wartościach bezwzględnych.
